# Query sui tre database, baseline e indici
Richiede il caricamento da 100k completato nel notebook 02. Si riusa un solo client per sessione; MongoClient ha un pool dati limitato a uno e socket di monitoraggio propri. Q1 recupera un record completo, Q2 conta JFK, Q3 conta distanze 1500–2000 incluse, Q4 calcola AVG per compagnia ignorando soltanto i null.
Prima misuriamo Q1–Q4 con sola PK. Poi creiamo e misuriamo separatamente i due indici e ripetiamo Q2/Q3. Le query restituiscono risultati logici equivalenti fra i tre adapter, senza imporre SQL a MongoDB.
`ABILITA_BENCHMARK=False` disabilita snapshot, benchmark e modifiche degli indici. Questo percorso manuale da 100K serve allo studio delle singole fasi; per nuove campagne usare esclusivamente `04_full_runner.ipynb`, senza alternarli sulla stessa destinazione. Per mostrare dati già popolati usare la DEMO, non questo notebook.


In [ ]:
from pathlib import Path
import csv
import json
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_config, selected_dataset
from src.databases.factory import create_adapter
from src.query_validation import build_reference, capture_database_snapshot, compare_postgres_snapshot, read_postgres_snapshot, read_mongo_snapshot, compare_reference_snapshots
from src.results import ResultRun
from src.benchmark import benchmark_phase, measure_secondary_indexes
config = load_config(ROOT / 'config' / 'benchmark.yaml')
print(json.dumps(config.manifest(), indent=2))

ABILITA_BENCHMARK = False
if ABILITA_BENCHMARK is True and config.dataset_size != 100000:
    raise ValueError('Percorso manuale limitato a 100.000 record: usare il runner per altre dimensioni autorizzate.')
if ABILITA_BENCHMARK is True:
    csv_path, artifact = selected_dataset(config)
    reference = build_reference(csv_path, config.dataset_size, artifact['sha256'], config.q1_ids, config.parameters)
    print('Attesi dal CSV: Q2=', reference['Q2'], 'Q3=', reference['Q3'], 'gruppi Q4=', len(reference['Q4']))

## Riferimenti sequenziali PostgreSQL e Mongo
Su PostgreSQL o Mongo già popolato, configurazione + questa cella esportano uno snapshot verificato senza cambiare dati/indici. Fermati prima delle misure, arresta il DB e passa al successivo. Imposta i percorsi stampati in `comparison.postgres_snapshot` e `comparison.mongo_snapshot`. CockroachDB richiede entrambi e li confronta prima e dopo gli indici; nessun DB concorrente è necessario.

In [ ]:
if ABILITA_BENCHMARK is True:
    if config.database in ('postgres', 'mongo'):
        with ResultRun(config.results_dir, config.manifest(), reference, stage='comparison_snapshot') as snapshot_run:
            source_adapter = create_adapter(config)
            try:
                source_adapter.connect()
                snapshot_run.set_database_version(source_adapter.version())
                state = 'primary_only' if len(source_adapter.indexes()) == 1 else 'secondary'
                snapshot_run.save_json('database_snapshot.json', capture_database_snapshot(source_adapter, reference, state))
            finally:
                source_adapter.close()
        print('Snapshot', config.database, ':', snapshot_run.directory / 'database_snapshot.json')
    else:
        read_postgres_snapshot(reference, config.postgres_snapshot)
        read_mongo_snapshot(reference, config.mongo_snapshot)
        print('Snapshot PostgreSQL e Mongo selezionati:', config.postgres_snapshot, config.mongo_snapshot)
else:
    print('Operazione disabilitata: ABILITA_BENCHMARK=False.')


## Warm-up, misure e rifiuto dei risultati scorretti
Default: 3 warm-up e 20 prove per query/scenario, configurabili in YAML. I warm-up sono salvati ma esclusi da media, mediana, deviazione standard campionaria, minimo e massimo. Nessun outlier viene eliminato. Q1 usa ID diversi, con sequenza e seed salvati.
Il timer `perf_counter_ns` include invio, acquisizione e decodifica completa del risultato; esclude confronto con il CSV, digest, salvataggio e visualizzazione. Prima di ogni scenario si verificano schema, righe/ID/null, campioni Q1, Q2/Q3, tutte le medie Q4 e gli indici reali. Una differenza interrompe il run.
I piani sono raccolti separatamente: JSON PostgreSQL, executionStats Mongo e testo nativo EXPLAIN ANALYZE Cockroach. Non forziamo il planner. I secondari Cockroach includono implicitamente la PK e la loro creazione attende lo stato pubblico. Single-node non misura disponibilità o scalabilità distribuita. I timeout 120s non sono aumentati: audit su dimensioni maggiori possono superarli; nessun adeguamento automatico.

## Riesecuzione consapevole
Se gli indici secondari esistono già, il default si ferma senza cancellarli. Per ripetere esplicitamente la baseline impostare `benchmark.confirm_drop_indexes: true`, verificare schema/tabella e riportare poi il flag a `false`. Questa rimozione riguarda soltanto i due indici del laboratorio, non i record. Non è la modalità DEMO.

In [ ]:
if ABILITA_BENCHMARK is True:
    with ResultRun(config.results_dir, config.manifest(), reference, stage='queries') as run:
        adapter = create_adapter(config)
        try:
            adapter.connect()
            run.set_database_version(adapter.version())
            if config.confirm_drop_indexes:
                adapter.drop_secondary_indexes(confirm_drop=True)
            adapter.validate_schema()
            adapter.assert_index_state('primary_only')
            if config.database in ('postgres', 'cockroach'):
                adapter.analyze()
            if config.database == 'mongo':
                run.save_json('postgres_comparison_primary_only.json', compare_postgres_snapshot(adapter, reference, config.postgres_snapshot, 'primary_only'))
            elif config.database == 'cockroach':
                run.save_json('reference_comparison_primary_only.json', compare_reference_snapshots(adapter, reference, config.postgres_snapshot, config.mongo_snapshot, 'primary_only'))
            benchmark_phase(adapter, reference, run, index_state='primary_only', query_ids=('Q1', 'Q2', 'Q3', 'Q4'), warmup=config.warmup, repetitions=config.repetitions)
            measure_secondary_indexes(adapter, run)
            benchmark_phase(adapter, reference, run, index_state='secondary', query_ids=('Q2', 'Q3'), warmup=config.warmup, repetitions=config.repetitions)
            if config.database == 'mongo':
                run.save_json('postgres_comparison_secondary.json', compare_postgres_snapshot(adapter, reference, config.postgres_snapshot, 'secondary'))
            elif config.database == 'cockroach':
                run.save_json('reference_comparison_secondary.json', compare_reference_snapshots(adapter, reference, config.postgres_snapshot, config.mongo_snapshot, 'secondary'))
        finally:
            adapter.close()
    print('Run:', run.run_id, '| directory:', run.directory)
else:
    print('Operazione disabilitata: ABILITA_BENCHMARK=False.')


## Risultati reali della sessione
Il riepilogo seguente deriva dal CSV appena scritto, senza numeri precostituiti. Query, creazione indici e ingestione sono eventi separati. Il campo batch_size dei raw query è nullo perché non è pertinente al tempo della query; il run di ingestione registra il batch effettivo. Per valutare i piani aprire `plans_primary_only.json` e `plans_secondary.json` nella directory del run.
I due notebook creano directory di run distinte, associate tramite checksum/dimensione dataset e configurazione. I tre motori vengono eseguiti separatamente e confrontati tramite snapshot reali. Non vengono eseguite prove da 1M/FULL o modalità DEMO.

In [ ]:
if ABILITA_BENCHMARK is True:
    with (run.directory / 'summary.csv').open(encoding='utf-8', newline='') as stream:
        summary = list(csv.DictReader(stream))
    for row in summary:
        print(row['event'], row['query_id'], row['index_state'], '| n=', row['valid_repetitions'], '| media ms=', row['mean_ms'], '| mediana ms=', row['median_ms'], '| dev.std ms=', row['sample_stddev_ms'])
    print('Dati e indici persistono nel volume del DB selezionato.')
else:
    print('Operazione disabilitata: ABILITA_BENCHMARK=False.')
